<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/mcp-arxiv-hero.svg" align="center" width="35%">
</div>

<br>

# MCP IN PRACTICE

<br>

**About:** This notebook stitches together the pieces from Notebooks 01 and 02 into a complete, running application, traces a user query end-to-end through the code, and works through the design decisions that separate a demo tool-caller from a production one.

**Learning Goals:** Read the complete architecture of an MCP application, trace message flow across user, client, model, and server, apply tool-design principles to a new capability, extend the running system with a new tool, and understand which deployment shape fits which use case.

**Keywords:** mcp, architecture, tool-design, production, deployment, extending-systems

**Prerequisite Knowledge:** (1) Completed [Building MCP Servers](01_building_mcp_servers.ipynb), (2) Completed [Building MCP Clients](02_building_mcp_clients.ipynb), (3) Familiarity with Python command-line applications.

**Target User:** Developers moving from a working prototype to a system they will actually ship, deploy, or extend.


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 0: SYSTEM ARCHITECTURE](#Part_0)
> #### [PART 1: THE ARXIV CHATBOT END-TO-END](#Part_1)
> #### [PART 2: MESSAGE FLOW EXAMPLES](#Part_2)
> #### [PART 3: TOOL DESIGN AND EXTENSION](#Part_3)
> #### [PART 4: DEPLOYMENT PATTERNS](#Part_4)

#### APPENDIX

> #### [COMPANION MATERIALS](#Appendix_1)
> #### [REFERENCES AND FURTHER READING](#Appendix_2)

<br>


<a id='Part_0'></a>

<hr style="border: 2px solid#003262;" />

#### PART 0

## **SYSTEM** ARCHITECTURE


<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/architecture.svg" align="center" width="55%" padding="10"><br>
    <br>
    Three layers: user (input/output), client (loop and state), server (tools and dispatch).
</div>

<br>

An MCP application separates cleanly into three layers, even when they live in the same file:

- **User layer** - the interface the human interacts with. A terminal prompt, a Slack channel, an HTTP endpoint. Its job is to collect queries and display answers, nothing else.
- **Client layer** - the loop that drives the conversation. It owns `messages`, dispatches tool calls, and decides when a response is terminal. The bulk of Notebook 02 lives here.
- **Server layer** - the tools themselves, their schemas, and the dispatcher. The bulk of Notebook 01 lives here.

In `arxiv_chatbot.py` all three layers share a single Python process, but they are logically independent. You can lift the server layer into a separate `server.py` file, then use it from a FastAPI-backed HTTP client (`examples/fastapi_arxiv_server.py`) or a Slack event handler with no code change to the tools themselves.

<br>

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/message-flow.svg" align="center" width="65%" padding="10"><br>
    <br>
    A single query can trigger multiple round-trips as tools are called and results fed back.
</div>


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **THE** ARXIV CHATBOT **END-TO-END**


This section walks the complete `arxiv_chatbot.py` module. Read each cell in order; the same code lives in the script and this is your annotated reading of it.

#### **1.1 Imports and environment**
___


In [ ]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

PAPER_DIR = "papers"

load_dotenv()
API_KEY = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=API_KEY) if API_KEY else None

print("Environment initialized" if client else "No API key - live calls disabled.")


All imports live at the top. `PAPER_DIR` is a module-level constant so every tool that reads or writes paper metadata sees the same location. Configuring `client` at import time means every function in the module can reuse it.


#### **1.2 Server layer - tools**
___


In [ ]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search arXiv and persist metadata for each hit."""
    client_arxiv = arxiv.Client()
    search = arxiv.Search(query=topic, max_results=max_results, sort_by=arxiv.SortCriterion.Relevance)
    papers = client_arxiv.results(search)

    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    file_path = os.path.join(path, "papers_info.json")

    try:
        with open(file_path, "r") as f:
            papers_info = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    paper_ids = []
    for paper in papers:
        pid = paper.get_short_id()
        paper_ids.append(pid)
        papers_info[pid] = {
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "summary": paper.summary,
            "pdf_url": paper.pdf_url,
            "published": str(paper.published.date()),
        }

    with open(file_path, "w") as f:
        json.dump(papers_info, f, indent=2)
    return paper_ids


def extract_info(paper_id: str) -> str:
    """Look up stored metadata for one paper by arXiv short ID."""
    for topic_dir in os.listdir(PAPER_DIR):
        path = os.path.join(PAPER_DIR, topic_dir)
        if not os.path.isdir(path):
            continue
        info_file = os.path.join(path, "papers_info.json")
        if not os.path.isfile(info_file):
            continue
        try:
            with open(info_file, "r") as f:
                papers_info = json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            continue
        if paper_id in papers_info:
            return json.dumps(papers_info[paper_id], indent=2)
    return f"There's no saved information related to paper {paper_id}."


#### **1.3 Server layer - schemas and dispatcher**
___


In [ ]:
tools = [
    {
        "name": "search_papers",
        "description": "Search arXiv for academic papers on a topic. Returns a list of paper IDs.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "Topic to search for."},
                "max_results": {"type": "integer", "description": "Maximum results.", "default": 5},
            },
            "required": ["topic"],
        },
    },
    {
        "name": "extract_info",
        "description": "Retrieve stored metadata for one paper by its arXiv short ID.",
        "input_schema": {
            "type": "object",
            "properties": {"paper_id": {"type": "string", "description": "arXiv short ID."}},
            "required": ["paper_id"],
        },
    },
]

tool_functions = {"search_papers": search_papers, "extract_info": extract_info}


def execute_tool(tool_name: str, tool_args: dict) -> str:
    result = tool_functions[tool_name](**tool_args)
    if result is None:
        return "The operation completed but did not return any results."
    if isinstance(result, list):
        return ", ".join(str(x) for x in result)
    if isinstance(result, dict):
        return json.dumps(result, indent=2)
    return str(result)


#### **1.4 Client layer - process_query**
___


In [ ]:
def process_query(query: str) -> None:
    """
    Drive one user query through the model. Prints text responses to stdout;
    invokes execute_tool for tool_use blocks.
    """
    messages = [{"role": "user", "content": query}]

    response = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=2048,
        tools=tools,
        messages=messages,
    )

    while True:
        assistant_blocks = list(response.content)
        made_tool_call = False

        for block in response.content:
            if block.type == "text":
                print(block.text)
            elif block.type == "tool_use":
                made_tool_call = True
                messages.append({"role": "assistant", "content": assistant_blocks})

                result = execute_tool(block.name, block.input)
                messages.append({
                    "role": "user",
                    "content": [{"type": "tool_result", "tool_use_id": block.id, "content": result}],
                })

                response = client.messages.create(
                    model="claude-3-7-sonnet-20250219",
                    max_tokens=2048,
                    tools=tools,
                    messages=messages,
                )
                break

        if not made_tool_call:
            return


#### **1.5 User layer - chat_loop**
___


In [ ]:
def chat_loop() -> None:
    """Terminal REPL. Ctrl-C exits cleanly; per-query exceptions are logged."""
    print("Type your queries or 'quit' to exit.\n")
    while True:
        try:
            query = input("Query: ").strip()
            if query.lower() == "quit":
                break
            process_query(query)
            print()
        except KeyboardInterrupt:
            print("\nInterrupted.")
            break
        except Exception as e:
            print(f"Error: {e}")


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **MESSAGE** FLOW **EXAMPLES**


Reading the code tells you *what* happens; tracing an example tells you *when*. Below are two traces at the API level - the exact sequence of `messages.create` calls and their arguments.

#### **2.1 Simple case - single search**

User query: `"Find papers about natural language processing"`

```
Round 1 request
  messages = [{"role": "user", "content": "Find papers about natural language processing"}]

Round 1 response
  content = [ToolUseBlock(name="search_papers", input={"topic": "..."}, id="toolu_1")]

Round 2 request
  messages = [
    {"role": "user", "content": "Find papers..."},
    {"role": "assistant", "content": [ToolUseBlock(...)]},
    {"role": "user", "content": [{"type": "tool_result", "tool_use_id": "toolu_1", "content": "2501.11111, 2501.22222, ..."}]},
  ]

Round 2 response
  content = [TextBlock(text="I found 5 papers on NLP. The most relevant are...")]

Terminal - no more tool_use blocks.
```

Two API calls, one tool invocation. This is the shortest useful shape.


#### **2.2 Multi-step case - search then extract**

User query: `"Find papers about transformers and tell me about the first one"`

The model recognizes the query needs two operations:

```
Round 1 request  -> Round 1 response: tool_use search_papers
Round 2 request  -> Round 2 response: tool_use extract_info (with the top ID)
Round 3 request  -> Round 3 response: text (final synthesis)
```

Three API calls, two tool invocations. The client loop iterates twice through its main body before returning.

Notice that the client code did *not* orchestrate the two-step plan; the model did. The client only knows how to route one response at a time. This is the pattern's strength: adding a new tool never requires new orchestration code.


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** For a query that requires *N* tool calls in sequence, how many times does the client call `client.messages.create`? Write the general formula and explain it in one sentence.

<br>

```python
# Fill in the formula and the sentence.
formula = "..."
explanation = "..."
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **TOOL** DESIGN AND **EXTENSION**


Good tool design is what separates a chatbot that mostly works from one that behaves predictably. Four principles apply.

- **Clear single purpose.** `search_papers` searches and stores. It does not also filter, delete, or re-rank. A tool with a compound purpose is a tool the model will use incorrectly.
- **Return types the model can consume.** Lists, JSON strings, and plain text. Never return an object whose `__str__` is unhelpful.
- **Graceful failure through return values.** If the paper is missing, return "no such paper" - do not raise. The dispatcher does not catch domain exceptions; the model does not read stack traces.
- **Idempotence where feasible.** `search_papers` is safe to call twice with the same topic - the second call overwrites the JSON with the current results. This matters because the model may retry on transient errors.

**Anti-patterns to avoid:**

- **Silent side effects.** A tool whose description says "search" but that also modifies unrelated state.
- **Unbounded output size.** A tool that returns 100 KB of text fills the model's context window immediately.
- **Overloaded parameters.** A `query` parameter that accepts a comma-separated string of five different things.
- **Hidden ordering dependencies.** A tool that requires another tool to have been called first, without saying so in the description.


#### **3.1 Extending the system - add a `save_papers_to_file` tool**
___

The three-step extension pattern applies to any new tool.

**Step 1**: write the function.


In [ ]:
def save_papers_to_file(paper_ids: str, filename: str) -> str:
    """
    Persist a list of paper IDs to a text file.

    Args:
        paper_ids: Comma-separated IDs (matches the dispatcher's list-to-string
                   normalization from search_papers).
        filename: Output file name; treated as a path relative to the current
                  working directory. Callers should ensure it is inside a safe
                  directory.

    Returns:
        Success message or a string starting with "Error".
    """
    try:
        with open(filename, "w") as f:
            f.write(paper_ids)
        count = len([p for p in paper_ids.split(",") if p.strip()])
        return f"Saved {count} paper IDs to {filename}"
    except Exception as e:
        return f"Error saving to {filename}: {e}"


**Step 2**: write the schema.


In [ ]:
save_tool_schema = {
    "name": "save_papers_to_file",
    "description": (
        "Save a comma-separated list of arXiv paper IDs to a text file. "
        "Use after search_papers when the user asks to persist results."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "paper_ids": {"type": "string", "description": "Comma-separated arXiv IDs (as returned by search_papers)."},
            "filename": {"type": "string", "description": "Destination filename (e.g., 'nlp_papers.txt')."},
        },
        "required": ["paper_ids", "filename"],
    },
}


**Step 3**: register with the dispatcher.


In [ ]:
tool_functions["save_papers_to_file"] = save_papers_to_file
tools.append(save_tool_schema)

print(f"Dispatcher now has {len(tool_functions)} tools: {list(tool_functions)}")


No change to `process_query` or `chat_loop`. The client layer is generic; the dispatcher is data-driven. This is the payoff for the layered design.

___

**Note:** The `filename` parameter above accepts an arbitrary path. In production, restrict it to a known directory (e.g., prepend a `results/` prefix inside the function) to prevent the model from writing outside your project.

___


<!--Concept Check banner-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left">
            <strong>CONCEPT</strong> CHECK
        </a>
    </span>
</div>
<!-------------------------------------->


> **Question:** The `save_papers_to_file` tool above accepts any filename. What is a specific risky call the model could make, and how would you tighten the function so that call becomes safe? Rewrite the function.

<br>

```python
def save_papers_to_file_safe(paper_ids: str, filename: str) -> str:
    ### YOUR CODE HERE ###
    ...
```

<hr style="border: 2px solid#003262;" />


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **DEPLOYMENT** PATTERNS


The `process_query` function is intentionally decoupled from any specific front-end. That decoupling makes four deployment shapes each a small change to the *user layer*, not a rewrite.

#### **4.1 CLI chatbot (what this repo ships)**

- User runs `python arxiv_chatbot.py`.
- Interactive REPL: query in, answer out.
- Good for personal use, development, and unit-testing the tool chain.
- File: `arxiv_chatbot.py` in this repo.

#### **4.2 Web API (FastAPI)**

- Wrap `process_query` in a FastAPI `POST /query` endpoint that accepts `{"text": "...", "session_id": "..."}` and returns the final text.
- Per-session `messages` list stored server-side (in Redis, a database, or an in-memory dict keyed by `session_id`).
- Good for web frontends, mobile apps, browser extensions.
- Example: [`examples/fastapi_arxiv_server.py`](examples/fastapi_arxiv_server.py).

#### **4.3 Async and streaming**

- Async: run tool calls concurrently when the model requests several in one response.
- Streaming: emit tokens to the user as they arrive rather than waiting for the full response.
- Good for latency-sensitive UIs.
- Examples: [`examples/async_arxiv_chatbot.py`](examples/async_arxiv_chatbot.py), [`examples/streaming_arxiv_chatbot.py`](examples/streaming_arxiv_chatbot.py).

#### **4.4 Reflection and self-critique**

- After the terminal text response, the client sends a second query asking the model to critique its own answer, then either returns the refined answer or the original.
- Increases quality at the cost of latency and tokens.
- Example: [`examples/multi_tool_reflection.py`](examples/multi_tool_reflection.py).

All four use the same `tool_functions` registry and the same schemas. That is the entire point of layering.


#### **4.5 Production checklist**
___

Before running any of the above with real users:

- **Log every tool call.** Name, arguments, and result. This is your only debuggability once the model is choosing tools autonomously.
- **Bound token usage.** Set `max_tokens` on every `messages.create` call and cap the number of iterations in `process_query`.
- **Sanitize tool arguments at the boundary.** File paths, SQL fragments, URLs - do not trust arguments blindly just because the schema says they are strings.
- **Handle rate limits explicitly.** Both the model API and any downstream service (arXiv, your database).
- **Track cost per session.** Multi-round tool loops can send the full history dozens of times; the token bill compounds.


<!--Navigate back to table of contents-->
<div align="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents">
            Table of Contents
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Appendix_1'></a>

<hr style="border: 2px solid#003262;" />

#### APPENDIX

## **COMPANION** MATERIALS


- **Homework (capstone)**: [MCP in Practice Homework](homework/mcp_in_practice_homework.ipynb) - design and build your own MCP application in a new domain, applying every principle from this notebook.
- **Working script**: `arxiv_chatbot.py` - the reference implementation.
- **Advanced examples**: [`examples/`](examples/) - production-shaped variants: async, streaming, FastAPI, reflection.
- **Previous notebooks**: [Building MCP Servers](01_building_mcp_servers.ipynb), [Building MCP Clients](02_building_mcp_clients.ipynb).


<a id='Appendix_2'></a>

<hr style="border: 2px solid#003262;" />

##### **REFERENCES**


- [Model Context Protocol specification](https://modelcontextprotocol.io/)
- [Anthropic Messages API - Tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [FastAPI documentation](https://fastapi.tiangolo.com/)
- [arxiv Python client](https://github.com/lukasschwab/arxiv.py)


<hr style="border: 6px solid#003262;" />
